In [ ]:
from datasets import load_dataset
import re
# Group by repository
from collections import defaultdict

In [ ]:
import torch.multiprocessing as mp
import multiprocessing


In [ ]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

In [ ]:
try:
    multiprocessing.set_start_method("spawn", force=True)
    mp.set_start_method("spawn", force=True)
    print("Yay")
except RuntimeError:
    # The start method has already been set.
    pass


In [ ]:
from retirever_v2 import main

In [ ]:
import json

def load_jsonl(filename):
    """Load JSONL file - each line is a separate JSON object"""
    data = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            # Each line is a complete JSON object
            data.append(json.loads(line.strip()))
    return data




In [ ]:
actual_data = load_jsonl("/home/ec2-user/cceval_data/python/line_completion.jsonl")

In [ ]:
len(actual_data)

In [ ]:
actual_data[100]

In [ ]:
print(actual_data[653]['prompt'])

In [ ]:
print(actual_data[0]['groundtruth'])

In [ ]:


repo_groups = defaultdict(list)
for item in actual_data:
    repo = item['metadata']['repository']
    repo_groups[repo].append(item)

# Sort repositories by number of items (descending)
sorted_repos = sorted(repo_groups.items(), key=lambda x: len(x[1]), reverse=True)



In [ ]:
len(sorted_repos)

In [ ]:
for repo_key, examples in (
                sorted_repos
            ):
        print(repo_key)
        print(examples)
        break
    

In [ ]:
examples[0]

In [ ]:
sorted_repos[0][1][0]  # Display top 10 repositories by item count

In [ ]:
i=0
for i in range(len(sorted_repos)):
    i+=1
    if len(sorted_repos[i][1])<5:
        print(i) 
        break   

In [ ]:
actual_data[0]['metadata'].keys()

<!-- ## OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO -->

In [ ]:
from retirever_v2 import stage_1
from langchain_neo4j import Neo4jGraph
from transformers import AutoTokenizer, AutoModel
from CrossCodeEval_end_to_end import create_graph_subprocess
import torch


In [ ]:
i=0
for x,y in sorted_repos:
    if x=='Adeon18-Mori-Bazius-Backend-f33b8ba':
        break
    i+=1
        

In [ ]:
i

In [ ]:
index=10
repo_index=1


In [ ]:
sorted_repos[repo_index]

In [ ]:
print(sorted_repos[repo_index][1][index]['prompt'])

In [ ]:
file_name=sorted_repos[repo_index][1][index]['metadata']['file']

In [ ]:
prompt_lines = sorted_repos[repo_index][1][index]['prompt'].split("\n")
retrieval_prompt = "\n".join(prompt_lines[-10:])

In [ ]:
query_format = """
Given repo_name:{repo_name}
Given file_name:{file_name}
Fetch the most important connected nodes from the graph to predict the next line of the below code:
{code}"""

In [ ]:
url = "bolt://localhost:7687"
username = "neo4j"
password = "nutanix_neo4j"
graph = Neo4jGraph(url=url, username=username, password=password)

In [ ]:
create_graph_subprocess("/home/ec2-user/CrossCodeEval_repos/hq0709_Depth-NeuS", "", {"url": "bolt://localhost:7687", "username": "neo4j", "password": "nutanix_neo4j"}, graph)

In [ ]:
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
print(f"Loading {EMBEDDING_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, padding_side="left")
model = AutoModel.from_pretrained(
    EMBEDDING_MODEL, torch_dtype=torch.float16, trust_remote_code=True
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
model.eval()

In [ ]:
query = query_format.format(
            repo_name='/home/ec2-user/CrossCodeEval_repos/hq0709_Depth-NeuS', file_name=file_name.replace('.py', '').replace('/', '.'), code=sorted_repos[repo_index][1][index]['prompt']
        )
# Get retrieved results using stage_1


In [ ]:
prompt=sorted_repos[repo_index][1][index]['prompt']

In [ ]:
if len(tokenizer.encode(query)) > 3000:
            # Split prompt into lines
            lines = prompt.split("\n")

            # Get all import lines
            import_lines = [line for line in lines if "import" in line]
            import_text = "\n".join(import_lines)

            # Calculate remaining token budget
            base_query = query_format.format(
                repo_name='/home/ec2-user/CrossCodeEval_repos/hq0709_Depth-NeuS',
                file_name=file_name.replace(".py", "").replace("/", "."),
                code="",
            )
            base_tokens = len(tokenizer.encode(base_query))
            import_tokens = len(tokenizer.encode(import_text))
            remaining_tokens = 3000 - base_tokens - import_tokens

            # Add bottom lines until we hit the token limit
            bottom_lines = []
            for line in reversed(lines):
                if "import" not in line:  # Skip import lines since we already have them
                    test_bottom = "\n".join(reversed(bottom_lines + [line]))
                    if len(tokenizer.encode(test_bottom)) < remaining_tokens:
                        bottom_lines.append(line)
                    else:
                        break

            # Combine imports + bottom lines
            bottom_text = "\n".join(reversed(bottom_lines))
            truncated_code = import_text + "\n...\n" + bottom_text

            # Recreate query with truncated code
            query = query_format.format(
                repo_name='/home/ec2-user/CrossCodeEval_repos/hq0709_Depth-NeuS',
                file_name=file_name.replace(".py", "").replace("/", "."),
                code=truncated_code,
            )

In [ ]:
print(query)

In [ ]:
retrieved_results = stage_1(graph, query, model, tokenizer)

In [ ]:
retrieved_results

In [ ]:
# from typing import List, Dict

# def format_retrieved_context(
#     retrieved_results: List[Dict],
#     code_tokenizer,
#     prompt: str,
#     max_context_tokens: int = 4096,
# ) -> str:
#     """
#     Formats retrieved code structure as comments and prepends to the prompt,
#     respecting a token limit for the context part only.

#     The format is:
#     # Module: <module_name>
#     #   Class: <class_name>
#     #     - <method_signature>
#     #   Functions:
#     #     - <function_signature>

#     Args:
#         retrieved_results: List of retrieved code entities.
#         code_tokenizer: Tokenizer to count tokens.
#         prompt: The original prompt.
#         max_context_tokens: Maximum tokens for the generated context comments.
#                             The prompt's own token count is not part of this limit.

#     Returns:
#         String with retrieved context as comments prepended to the prompt.
#     """
#     if not retrieved_results:
#         return prompt

#     # 1. Group results by module and then by class
#     grouped = defaultdict(lambda: defaultdict(list))
#     for item in retrieved_results:
#         module_name = item.get("module_name", "unknown_module")
#         class_name = item.get("class")
#         if class_name:
#             grouped[module_name][class_name].append(item)
#         else:
#             grouped[module_name]["<standalone_functions>"].append(item)

#     grouped_results = {k: dict(v) for k, v in grouped.items()}

#     # 2. Build the context string, respecting the token limit for the context itself
#     header = "# Here is a summary of relevant code from other files:"
#     context_parts = [header]
    
#     current_tokens = len(code_tokenizer.encode(header))

#     for module_name, entities in grouped_results.items():
#         # Create the string for the current module block
#         module_block_lines = [f"\n# Module: {module_name}"]
        
#         sorted_entities = sorted(entities.items(), key=lambda item: item[0] == '<standalone_functions>')

#         for entity_name, items in sorted_entities:
#             if entity_name == '<standalone_functions>':
#                 if not items: continue
#                 module_block_lines.append(f"#   Functions/Classes:")
#                 for func in items:
#                     display = func.get('signature',"")
#                     if display is None or len(display)==0:
#                         display=func.get('name', 'N/A')
#                     module_block_lines.append(f"#     - {display}")
#             else:  # It's a class
#                 module_block_lines.append(f"#   Class: {entity_name}")
#                 for method in items:
#                     display = method.get('signature',"")
#                     if display is None or len(display)==0:
#                         display=method.get('name', 'N/A')
#                     module_block_lines.append(f"#     - {display}")

#         module_block = "\n".join(module_block_lines)
#         module_tokens = len(code_tokenizer.encode(module_block))

#         # Check if adding this module block would exceed the context limit
#         if current_tokens + module_tokens > max_context_tokens:
#             print(f"Context token limit ({max_context_tokens}) reached. Stopping context generation.")
#             break
        
#         context_parts.append(module_block)
#         current_tokens += module_tokens

#     # 3. Combine context and original prompt
#     if len(context_parts) == 1:  # Only header was added, no context fit
#         return prompt

#     context_str = "".join(context_parts)
#     return context_str + "\n\n" + prompt

In [ ]:
# formatted_prompt = format_retrieved_context(retrieved_results, tokenizer, sorted_repos[repo_index][1][index]['prompt'])

In [ ]:
# print(formatted_prompt)

In [ ]:
from CrossCodeEvaluation import generate_with_code_model, compute_exact_match, compute_edit_similarity

In [ ]:
# res=generate_with_code_model(formatted_prompt)
# print(res)

In [ ]:
# sorted_repos[repo_index][1][index]['groundtruth']

In [ ]:
# compute_exact_match(sorted_repos[repo_index][1][index]['groundtruth'], res)

In [ ]:
# compute_edit_similarity(res, sorted_repos[repo_index][1][index]['groundtruth'])

In [ ]:
# retrieved_results[0]['labels(dep)'][0]

In [ ]:
# graph.query("MATCH (m:Module {name: $module_name})-[:CONTAINS*]->(children) RETURN children.name, children.signature, children.code",{"module_name": retrieved_results[0]['dep']['name']})

In [ ]:
import json

with open('cceval_results_tracking_bm25_codellama_7b.json', 'r') as f:
    data = json.load(f)

total_em_weighted = 0
total_es_weighted = 0
total_examples = 0

for repo_name, metrics in data.items():
    examples = metrics['total_examples']
    total_em_weighted += metrics['exact_match_rate'] * examples
    total_es_weighted += metrics['average_edit_similarity'] * examples
    total_examples += examples

# Weighted averages
weighted_avg_em = total_em_weighted / total_examples
weighted_avg_es = total_es_weighted / total_examples

print(f"Total examples: {total_examples}")
print(f"Weighted Average Exact Match Rate: {weighted_avg_em:.4f}")
print(f"Weighted Average Edit Similarity: {weighted_avg_es:.4f}")

In [ ]:
import json

with open('cceval_results_tracking_hybrid_deepseek_6_7b_final_3.json', 'r') as f:
    data = json.load(f)
i=0
total_em_weighted = 0
total_es_weighted = 0
total_examples = 0
repo_names_processed=[]
for repo_name, metrics in data.items():

    
    repo_names_processed.append(repo_name)
    examples = metrics['total_examples']
    total_em_weighted += metrics['exact_match_rate'] * examples
    total_es_weighted += metrics['average_edit_similarity'] * examples
    total_examples += examples
    i+=1

# Weighted averages
weighted_avg_em = total_em_weighted / total_examples
weighted_avg_es = total_es_weighted / total_examples

print(f"Total examples: {total_examples}")
print(f"Weighted Average Exact Match Rate: {weighted_avg_em:.4f}")
print(f"Weighted Average Edit Similarity: {weighted_avg_es:.4f}")

<!-- ## BM25 -->

In [ ]:
import json

with open('cceval_results_tracking_bm25_2.json', 'r') as f:
    data = json.load(f)
i=0
total_em_weighted = 0
total_es_weighted = 0
total_examples = 0
repo_names_processed=[]
for repo_name, metrics in data.items():

    
    repo_names_processed.append(repo_name)
    examples = metrics['total_examples']
    total_em_weighted += metrics['exact_match_rate'] * examples
    total_es_weighted += metrics['average_edit_similarity'] * examples
    total_examples += examples
    i+=1

# Weighted averages
weighted_avg_em = total_em_weighted / total_examples
weighted_avg_es = total_es_weighted / total_examples

print(f"Total examples: {total_examples}")
print(f"Weighted Average Exact Match Rate: {weighted_avg_em:.4f}")
print(f"Weighted Average Edit Similarity: {weighted_avg_es:.4f}")

In [ ]:
import json

with open('cceval_results_tracking_hybrid_codellama_7b.json', 'r') as f:
    data = json.load(f)
i=0
total_em_weighted = 0
total_es_weighted = 0
total_examples = 0
repo_names_processed=[]
for repo_name, metrics in data.items():

    
    repo_names_processed.append(repo_name)
    examples = metrics['total_examples']
    total_em_weighted += metrics['exact_match_rate'] * examples
    total_es_weighted += metrics['average_edit_similarity'] * examples
    total_examples += examples
    i+=1

# Weighted averages
weighted_avg_em = total_em_weighted / total_examples
weighted_avg_es = total_es_weighted / total_examples

print(f"Total examples: {total_examples}")
print(f"Weighted Average Exact Match Rate: {weighted_avg_em:.4f}")
print(f"Weighted Average Edit Similarity: {weighted_avg_es:.4f}")

In [ ]:
# repo_names_unprocessed=[]

In [ ]:
# for repo_key , examples in sorted_repos:
#     if repo_key not in repo_names_processed:
#         print(repo_key)
#         repo_names_unprocessed.append(repo_key)

In [ ]:
# with open('/home/ec2-user/CrossCodeEval/graphrag_evaluation/cceval_skipped_repos.txt', 'w') as f:
#     for repo_name in repo_names_unprocessed:

#         f.write(repo_name + '\n')

In [ ]:
# import subprocess
# import os
# import shutil
# def clone_repo_at_commit(repo_key, base_dir="/home/ec2-user/CrossCodeEval_repos/"):
#     """
#     Parse a repository key like 'turboderp-exllama-a544085', clone the repo, and checkout to the specified commit.

#     Args:
#         repo_key: String like 'turboderp-exllama-a544085'
#         base_dir: Base directory to store cloned repos

#     Returns:
#         tuple: (repo_path, error_message) where repo_path is None if failed
#     """
#     # Parse repo name and commit id
#     parts = repo_key.split("-")
#     if len(parts) < 3:
#         return None, f"Invalid repository key format: {repo_key}"
#     commit_id = parts[-1]
#     repo_name = None

#     # Strategy 1: Try standard owner/repo format (most common)
#     if len(parts) >= 3:
#         # For datasig-ac-uk-RoughPy-a544085 -> datasig-ac-uk/RoughPy
#         owner_parts = []
#         repo_parts = []

#         # Look for common patterns or try different splits
#         # First, try assuming last part is commit, second-to-last is repo name
#         potential_repo = parts[-2]
#         potential_owner = "-".join(parts[:-2])
#         repo_name = f"{potential_owner}/{potential_repo}"

#         # Fallback: if that doesn't work, try the original logic
#         if not potential_owner:
#             repo_name = f"{parts[0]}/{'-'.join(parts[1:-1])}"

#     try:
#         os.makedirs(base_dir, exist_ok=True)
#         clean_repo_name = repo_name.replace("/", "_")
#         repo_path = os.path.join(base_dir, clean_repo_name)
#         if os.path.exists(repo_path):
#             shutil.rmtree(repo_path)

#         print(f"  Cloning {repo_name}...")
#         clone_url = f"https://github.com/{repo_name}.git"
#         env = os.environ.copy()
#         env.update(
#             {
#                 "GIT_TERMINAL_PROMPT": "0",
#                 "GIT_ASKPASS": "echo",
#                 "SSH_ASKPASS": "echo",
#             }
#         )
#         try:
#             result = subprocess.run(
#                 ["git", "clone", clone_url, repo_path],
#                 capture_output=True,
#                 text=True,
#                 timeout=180,
#                 env=env,
#             )
#         except subprocess.TimeoutExpired:
#             return (
#                 None,
#                 f"Repository {repo_name} clone timed out (likely private or network issue)",
#             )

#         if result.returncode != 0:
#             error_msg = result.stderr.strip()
#             if (
#                 "repository not found" in error_msg.lower()
#                 or "not found" in error_msg.lower()
#             ):
#                 return None, f"Repository {repo_name} not found (404)"
#             elif (
#                 "permission denied" in error_msg.lower()
#                 or "forbidden" in error_msg.lower()
#             ):
#                 return None, f"Repository {repo_name} is private or access denied (403)"
#             elif "could not resolve host" in error_msg.lower():
#                 return None, f"Network error while cloning {repo_name}"
#             else:
#                 return None, f"Git clone failed for {repo_name}: {error_msg}"

#         os.chdir(repo_path)
#         print(f"  Checking out to commit: {commit_id}...")
#         result = subprocess.run(
#             ["git", "checkout", commit_id], capture_output=True, text=True
#         )
#         if result.returncode != 0:
#             error_msg = (
#                 result.stderr.strip() if result.stderr else "Unknown checkout error"
#             )
#             return None, f"Git checkout failed for commit {commit_id}: {error_msg}"

#         print(f"  ✓ Successfully prepared repository at {repo_path}")
#         return repo_path, None

#     except Exception as e:
#         return None, f"Unexpected error cloning repository {repo_name}: {e}"

In [ ]:
# for repo_name in repo_names_unprocessed:
#     res=clone_repo_at_commit(repo_name)
#     if res[0]:
#         print(repo_name)
#     else:
#         print(res[1])

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict
from rank_bm25 import BM25Okapi

In [ ]:
class PythonCodeBM25Searcher:
    """
    Simple BM25 index for Python code files using rank-bm25 library
    """

    def __init__(self, folder_path: str, max_lines_per_chunk: int = 10):
        """
        Initialize the BM25 searcher

        Args:
            folder_path: Path to folder containing Python files
            max_lines_per_chunk: Maximum lines per chunk (default: 10)
        """
        self.folder_path = Path(folder_path)
        self.max_lines_per_chunk = max_lines_per_chunk
        self.corpus = []
        self.chunk_metadata = []  # Store file info for each chunk
        self.bm25 = None

        # Build index on initialization
        self._build_index()

    def _get_python_files(self) -> List[Path]:
        """Get all Python files from the folder recursively"""
        return list(self.folder_path.rglob("*.py"))

    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenizer that splits on whitespace and removes special chars"""
        import string

        # Remove punctuation and split on whitespace
        text = text.translate(str.maketrans("", "", string.punctuation))
        return text.lower().split()

    def _chunk_code(self, code: str, file_path: Path) -> List[Dict]:
        """
        Chunk Python code into max_lines_per_chunk line segments

        Args:
            code: Python source code
            file_path: Path to the file

        Returns:
            List of chunk dictionaries with metadata
        """
        lines = code.split("\n")
        chunks = []

        for i in range(0, len(lines), self.max_lines_per_chunk):
            chunk_lines = lines[i : i + self.max_lines_per_chunk]
            chunk_text = "\n".join(chunk_lines).strip()

            if chunk_text:  # Only add non-empty chunks
                chunks.append(
                    {
                        "text": chunk_text,
                        "file_path": str(file_path),
                        "start_line": i + 1,
                        "end_line": min(i + self.max_lines_per_chunk, len(lines)),
                        "total_lines": len(lines),
                    }
                )

        return chunks

    def _build_index(self):
        """Build BM25 index from all Python files in the folder"""
        python_files = self._get_python_files()

        if not python_files:
            raise ValueError(f"No Python files found in {self.folder_path}")

        print(f"Found {len(python_files)} Python files. Building index...")

        for file_path in python_files:
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    code = f.read()

                # Chunk the code
                chunks = self._chunk_code(code, file_path)

                # Add chunks to corpus
                for chunk in chunks:
                    self.corpus.append(chunk["text"])
                    self.chunk_metadata.append(chunk)

            except Exception as e:
                print(f"Error processing {file_path}: {e}")
                continue

        if not self.corpus:
            raise ValueError("No code chunks were created")

        print(f"Created {len(self.corpus)} code chunks. Creating BM25 index...")

        # Tokenize corpus
        tokenized_corpus = [self._tokenize(doc) for doc in self.corpus]

        # Create BM25 index
        self.bm25 = BM25Okapi(tokenized_corpus)

        print("BM25 index created successfully!")

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """
        Search for most relevant code chunks

        Args:
            query: Search query
            top_k: Number of top results to return

        Returns:
            List of dictionaries containing chunk info and scores
        """
        if not self.bm25:
            raise ValueError("Index not built yet")

        # Tokenize query
        tokenized_query = self._tokenize(query)

        # Get scores for all documents
        scores = self.bm25.get_scores(tokenized_query)

        # Get top k indices and scores
        top_indices = scores.argsort()[-top_k:][::-1]

        search_results = []
        for idx in top_indices:
            if scores[idx] > 0:  # Only include results with positive scores
                result = {
                    "score": float(scores[idx]),
                    "text": self.corpus[idx],
                    "metadata": self.chunk_metadata[idx],
                }
                search_results.append(result)

        return search_results

    def search_excluding_file(
        self, query: str, exclude_file: str, top_k: int = 5
    ) -> List[Dict]:
        """
        Search for most relevant code chunks while excluding chunks from a specific file

        Args:
            query: Search query
            exclude_file: File path to exclude from results
            top_k: Number of top results to return

        Returns:
            List of dictionaries containing chunk info and scores
        """
        if not self.bm25:
            raise ValueError("Index not built yet")

        # Tokenize query
        tokenized_query = self._tokenize(query)

        # Get scores for all documents
        scores = self.bm25.get_scores(tokenized_query)

        # Get more candidates to filter from
        top_indices = scores.argsort()[-top_k * 3 :][::-1]

        search_results = []
        for idx in top_indices:
            if scores[idx] > 0:
                # Skip chunks from the excluded file
                chunk_file = self.chunk_metadata[idx]["file_path"]

                chunk_normalized_file = Path(chunk_file).as_posix()
                exclude_normalized = Path(exclude_file).as_posix()
                print(chunk_normalized_file, exclude_normalized)
                if not chunk_normalized_file.endswith(exclude_normalized):
                    result = {
                        "score": float(scores[idx]),
                        "text": self.corpus[idx],
                        "metadata": self.chunk_metadata[idx],
                    }
                    search_results.append(result)
                    if len(search_results) >= top_k:
                        break
                else:
                    print(f"Skipping chunk from {chunk_file} because it's the same as {exclude_file}")

        return search_results

    def clear_index(self):
        """
        Clear the built BM25 index and reset all data structures
        """
        self.corpus = []
        self.chunk_metadata = []
        self.bm25 = None
        print("BM25 index cleared successfully!")

In [ ]:
searcher=PythonCodeBM25Searcher("/home/ec2-user/CrossCodeEval_repos/hq0709_Depth-NeuS")

In [ ]:
results = searcher.search_excluding_file(retrieval_prompt, exclude_file=file_name, top_k=10)


In [ ]:
print(results[0]['text'])

In [ ]:
print(retrieval_prompt)

In [ ]:
searcher.clear_index()

In [ ]:
len(tokenizer.encode(results[0]['text']))

In [ ]:
def format_retrieved_context(
    retrieved_results: List[Dict],
    bm25_retrieved_results: List[str],
    code_tokenizer,
    prompt: str,
    max_context_tokens: int = 4096,
) -> str:
    """
    Formats retrieved code structure as comments and prepends to the prompt,
    respecting a token limit for the context part only.

    The format is:
    # Module: <module_name>
    #   Class: <class_name>
    #     - <method_signature>
    #   Functions:
    #     - <function_signature>

    Args:
        retrieved_results: List of retrieved code entities.
        bm25_retrieved_results: List of BM25 retrieved code chunks (strings).
        code_tokenizer: Tokenizer to count tokens.
        prompt: The original prompt.
        max_context_tokens: Maximum tokens for the generated context comments.
                            The prompt's own token count is not part of this limit.

    Returns:
        String with retrieved context as comments prepended to the prompt.
    """
    context_parts = []
    
    # 1. Process main retrieved_results (limit: 3800 tokens)
    if retrieved_results:
        # Group results by module and then by class
        grouped = defaultdict(lambda: defaultdict(list))
        for item in retrieved_results:
            module_name = item.get("module_name", "unknown_module")
            class_name = item.get("class")
            if class_name:
                grouped[module_name][class_name].append(item)
            else:
                grouped[module_name]["<standalone_functions>"].append(item)

        grouped_results = {k: dict(v) for k, v in grouped.items()}

        # Build the main context string, respecting the 3800 token limit
        header = "# Here is a summary of relevant code from other files:"
        main_context_parts = [header]

        current_tokens = len(code_tokenizer.encode(header))
        max_main_tokens = 3000

        for module_name, entities in grouped_results.items():
            # Create the string for the current module block
            module_block_lines = [f"\n# Module: {module_name}"]

            sorted_entities = sorted(
                entities.items(), key=lambda item: item[0] == "<standalone_functions>"
            )

            for entity_name, items in sorted_entities:
                if entity_name == "<standalone_functions>":
                    if not items:
                        continue
                    module_block_lines.append(f"#   Functions/Classes:")
                    for func in items:
                        display = func.get("signature", "")
                        if display is None or len(display) == 0:
                            display = func.get("name", "N/A")
                        module_block_lines.append(f"#     - {display}")
                else:  # It's a class
                    module_block_lines.append(f"#   Class: {entity_name}")
                    for method in items:
                        display = method.get("signature", "")
                        if display is None or len(display) == 0:
                            display = method.get("name", "N/A")
                        module_block_lines.append(f"#     - {display}")

            module_block = "\n".join(module_block_lines)
            module_tokens = len(code_tokenizer.encode(module_block))

            # Check if adding this module block would exceed the main context limit
            if current_tokens + module_tokens > max_main_tokens:
                print(
                    f"Main context token limit ({max_main_tokens}) reached. Stopping main context generation."
                )
                break

            main_context_parts.append(module_block)
            current_tokens += module_tokens

        if len(main_context_parts) > 1:  # More than just header was added
            context_parts.append("".join(main_context_parts))

    # 2. Process BM25 retrieved_results as simple text strings (limit: 300 tokens)
    if bm25_retrieved_results:
        bm25_header = "\n\n# Here are some structurally same code fragments:"
        bm25_context_parts = [bm25_header]

        bm25_current_tokens = len(code_tokenizer.encode(bm25_header))
        max_bm25_tokens = 1000

        for code_text in bm25_retrieved_results:
            # Convert the code chunk to comments (prefix each line with #)
            code_lines = code_text.split('\n')
            commented_code_lines = [f"# {line}" for line in code_lines]
            commented_code = '\n'.join(commented_code_lines)
            
            code_block = f"\n{commented_code}"+'\n#'
            code_block_tokens = len(code_tokenizer.encode(code_block))

            # Check if adding this code block would exceed the BM25 context limit
            if bm25_current_tokens + code_block_tokens > max_bm25_tokens:
                print(
                    f"BM25 context token limit ({max_bm25_tokens}) reached. Stopping BM25 context generation."
                )
                break

            bm25_context_parts.append(code_block)
            bm25_current_tokens += code_block_tokens



        if len(bm25_context_parts) > 1:  # More than just header was added
            context_parts.append("".join(bm25_context_parts))

    # 3. Combine all context parts and original prompt
    if not context_parts:
        return prompt

    context_str = "".join(context_parts)
    return context_str + "\n\n" + prompt

In [ ]:
bm25_retrieved_results=[result['text'] for result in results]

In [ ]:
formatted_prompt = format_retrieved_context(retrieved_results, bm25_retrieved_results, tokenizer, sorted_repos[repo_index][1][index]['prompt'])

In [ ]:
print(formatted_prompt)

In [ ]:
res=generate_with_code_model(formatted_prompt)
print(res)

In [ ]:
sorted_repos[repo_index][1][index]['groundtruth']